# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. We'll follow a structured approach: loading the data, reviewing metadata, extracting record sets, conducting exploratory data analysis, and visualizing results.

### Dataset Source
The dataset is provided via a Croissant schema at:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# If running in a fresh environment, ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and access main dataset schema via `mlcroissant`. The `Dataset` object provides convenient access to all top-level metadata attributes.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset schema
dataset = mlc.Dataset(url)

# Show dataset metadata summary
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.date_published}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview

Review available record sets, fields, and their `@id` values. Croissant organizes the tabular (and other) data into **record sets**, each referencing zero or more fields (columns/features). We'll list and examine all record sets, fields, and a sample of their metadata for deeper inspection.

In [ ]:
from pprint import pprint

# List all available record sets (`@id`) and get overview:
all_record_sets = dataset.metadata.record_sets
print(f"Found {len(all_record_sets)} record sets:")
for rs in all_record_sets:
    print(f"- @id: {rs.id}")
    print(f"   Name: {rs.name}")
    print(f"   Description: {rs.description}")

# For demonstration, pick the main data table record set --
# Here we pick the first one (update as needed for this dataset):
if all_record_sets:
    main_record_set = all_record_sets[0]
    print(f"\nSample fields for record set '@id': {main_record_set.id}\n")
    if main_record_set.fields:
        for field in main_record_set.fields:
            print(f"  Field @id: {field.id}")
            print(f"    Name: {field.name}")
            print(f"    Data type: {field.data_type}")
            print(f"    Description: {field.description}")
    else:
        print(f"  No fields listed for this record set.")
else:
    print("No record sets found in this dataset.")

## 3. Data Extraction

Load data from specific record sets into pandas DataFrames. All usage of identifiers must reference `@id` fields. We'll demonstrate using the primary record set discovered above.

In [ ]:
# Extract records for each record set into a DataFrame (by @id):
dataframes = {}
for rs in all_record_sets:
    print(f"Loading records for RecordSet @id: {rs.id}")
    records = list(dataset.records(record_set=rs.id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Shape: {df.shape}\n")
    else:
        print("  No records loaded. Skipping.\n")

# Display the first rows of the main record set's dataframe (if available):
if all_record_sets:
    main_rs_id = all_record_sets[0].id
    if main_rs_id in dataframes:
        print(f"Displaying first five rows for RecordSet '@id': {main_rs_id}")
        display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps: filtering numeric field values, normalization, and grouping by a categorical field. All fields are referenced by their `@id` for consistency with the Croissant schema.

In [ ]:
# --- Adjust these for your data by inspecting prior outputs ---
# Choose a main record set and a numeric field (by their @id):
main_rs = all_record_sets[0] if all_record_sets else None
if main_rs is not None and main_rs.id in dataframes:
    df = dataframes[main_rs.id]

    # Find first numeric-type field
    numeric_field = None
    for field in main_rs.fields:
        if field.data_type in ('Float', 'Integer', 'Number', 'schema:Float', 'schema:Integer', 'schema:Number'):
            if field.id in df.columns:
                numeric_field = field.id
                print(f"Using numeric field @id: {numeric_field}")
                break
    if not numeric_field:
        print("No numeric field found. Please check field data types and IDs.")
    else:
        # Filtering example: values above a threshold
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where '{numeric_field}' > {threshold:.2f} (mean): {len(filtered_df)} rows")

        # Normalization (z-score)
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a likely categorical/text field, e.g., the first non-numeric field
        group_field = None
        for field in main_rs.fields:
            if field.data_type in ('Text', 'schema:Text') and field.id in df.columns:
                group_field = field.id
                break
        if group_field is not None:
            grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Mean of '{numeric_field}' grouped by '{group_field}':")
            print(grouped.head())
        else:
            print("No group field found for aggregation.")
else:
    print("Main record set not available or not loaded.")

## 5. Visualization

Visualize the distributions or relationships between fields. Here is an example visualization: histogram of the chosen numeric field and a boxplot of the grouped/wrangled data (if available).

In [ ]:
import matplotlib.pyplot as plt

if main_rs is not None and main_rs.id in dataframes and numeric_field:
    df = dataframes[main_rs.id]
    # Histogram
    plt.figure(figsize=(7,4))
    df[numeric_field].dropna().hist(bins=30)
    plt.title(f"Histogram of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.grid(True, axis='y', alpha=0.3)
    plt.show()

    # Boxplot for grouped values (if group_field available)
    if 'group_field' in locals() and group_field and group_field in df.columns:
        plt.figure(figsize=(10,5))
        df.dropna(subset=[numeric_field, group_field]).boxplot(column=numeric_field, by=group_field, grid=False)
        plt.xticks(rotation=45, ha='right')
        plt.title(f"'{numeric_field}' by '{group_field}'")
        plt.suptitle('')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion

We've loaded the FAIR² mass spectrometry dataset, inspected its structure using Croissant schema `@id` conventions, extracted table records, and performed basic filtering, normalization, grouping, and visualization.

**Key points:**
- All data references were made using entity `@id`, ensuring clarity in field and record selection.
- With `mlcroissant`, the data structure and metadata are programmatically discoverable, facilitating automated data exploration pipelines.

Further work could include integrating annotation workflows, performing target-specific analyses (e.g., protein name based extraction), or scaling this template for batch processing of Croissant datasets.